In [1]:
import cv2
from facenet_pytorch import MTCNN
import os
import matplotlib.pyplot as plt

/home/kovan/anaconda3/envs/MCGaze/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.insert(0, "..")

In [3]:
frame_id = 0
person_num = 0
video_clip=None
video_clip_set = []
vid_len = len(os.listdir('frames'))
while frame_id < vid_len:
    frame = cv2.imread('frames/%d.jpg' % frame_id)
    w,h,c = frame.shape
    txt_path = 'result/labels/%d.txt' % frame_id
    f = open(txt_path, 'r')
    #遍历每一行 
    face_bbox = []
    for line in f.readlines():
        line = line.strip()
        line = line.split(' ')
        for i in range(len(line)):
            line[i] = eval(line[i])
            #将每一行的数据存入字典
        if line[0]==1:
            face_bbox.append([(line[1]),(line[2]),(line[3]),(line[4])])
    f.close()
    #按第一维排序
    if face_bbox is not None:
        face_bbox = sorted(face_bbox, key= lambda x :x[0])
        cur_person_num = len(face_bbox)
    else:
        cur_person_num = 0
    if cur_person_num != person_num :
        if video_clip==None:
            video_clip={'frame_id': [], 'person_num': cur_person_num}
            video_clip['frame_id'].append(frame_id)
            for i in range(cur_person_num):
                video_clip['p'+str(i)]=[face_bbox[i]]
        else:
            video_clip_set.append(video_clip)
            video_clip={'frame_id': [], 'person_num': cur_person_num}
            video_clip['frame_id'].append(frame_id)
            for i in range(cur_person_num):
                video_clip['p'+str(i)]=[face_bbox[i]]
    else:
        video_clip['frame_id'].append(frame_id)
        for i in range(cur_person_num):
                video_clip['p'+str(i)].append(face_bbox[i])
    person_num = cur_person_num
    frame_id += 1

video_clip_set.append(video_clip)

In [4]:
from mmdet.apis import init_detector
from mmdet.datasets.pipelines import Compose
import torch
from mmcv.parallel import collate, scatter
import numpy as np
model = init_detector(
        '../configs/multiclue_gaze/multiclue_gaze_r50_l2cs.py',
        '../ckpts/multiclue_gaze_r50_l2cs.pth',
        device="cuda:0",
        cfg_options=None,)
cfg = model.cfg


/home/kovan/anaconda3/envs/MCGaze/lib/python3.9/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


load checkpoint from local path: ../ckpts/multiclue_gaze_r50_l2cs.pth


In [5]:
print(cfg.data.test.pipeline[1:])
test_pipeline = Compose(cfg.data.test.pipeline[1:])

def load_datas(data, test_pipeline, datas):
    datas.append(test_pipeline(data))

[{'type': 'Resize', 'img_scale': (448, 448), 'keep_ratio': True}, {'type': 'RandomFlip', 'flip_ratio': 0.0}, {'type': 'Normalize', 'mean': [123.675, 116.28, 103.53], 'std': [58.395, 57.12, 57.375], 'to_rgb': True}, {'type': 'Pad', 'size_divisor': 32}, {'type': 'DefaultFormatBundle'}, {'type': 'Collect', 'keys': ['img']}]


In [ ]:
def infer(datas,model,clip,i):
    datas = sorted(datas, key=lambda x:x['img_metas'].data['filename']) # 按帧顺序 img名称从小到大
    datas = collate(datas, samples_per_gpu=len(frame_id)) # 用来形成batch用的
    datas['img_metas'] = datas['img_metas'].data
    datas['img'] = datas['img'].data
    datas = scatter(datas, ["cuda:0"])[0]
    with torch.no_grad():
        (det_bboxes, det_labels), det_gazes = model(
                return_loss=False,
                rescale=True,
                format=False,# 返回的bbox既包含face_bboxes也包含head_bboxes
                **datas)    # 返回的bbox格式是[x1,y1,x2,y2],根据return_loss函数来判断是forward_train还是forward_test.
    gaze_dim = det_gazes['gaze_score'].size(1)
    det_fusion_gaze = det_gazes['gaze_score'].view((det_gazes['gaze_score'].shape[0], 1, gaze_dim))
    clip['gaze_p'+str(i)].append(det_fusion_gaze.cpu().numpy())

max_len = 100
for clip in video_clip_set:
    frame_id = clip['frame_id']
    person_num = clip['person_num']
    for i in range(person_num):
        head_bboxes = clip['p'+str(i)]
        clip['gaze_p'+str(i)] = []
        datas = []
        for j,frame in enumerate(frame_id):
            cur_img = cv2.imread("frames/"+str(frame)+".jpg")
            w,h,_ = cur_img.shape
            for xy in head_bboxes[j]:
                xy = int(xy)
            head_center = [int(head_bboxes[j][1]+head_bboxes[j][3])//2,int(head_bboxes[j][0]+head_bboxes[j][2])//2]
            l = int(max(head_bboxes[j][3]-head_bboxes[j][1],head_bboxes[j][2]-head_bboxes[j][0])*0.8)
            head_crop = cur_img[max(0,head_center[0]-l):min(head_center[0]+l,w),max(0,head_center[1]-l):min(head_center[1]+l,h),:]
            w_n,h_n,_ = head_crop.shape
            # if frame==0:
            #     plt.imshow(head_crop)
            # print(head_crop.shape)
            cur_data = dict(filename=j,ori_filename=111,img=head_crop,img_shape=(w_n,h_n,3),ori_shape=(2*l,2*l,3),img_fields=['img'])
            load_datas(cur_data,test_pipeline,datas)
            if len(datas)>max_len or j==(len(frame_id)-1):
                infer(datas,model,clip,i)
                datas = []
                if j==(len(frame_id)-1):
                    clip['gaze_p'+str(i)] = np.concatenate(clip['gaze_p'+str(i)],axis=0)


torch.Size([101, 3, 448, 448])


/home/kovan/anaconda3/envs/MCGaze/lib/python3.9/site-packages/torch/_tensor.py:760: UserWarning: non-inplace resize is deprecated
  warnings.warn("non-inplace resize is deprecated")


torch.Size([101, 3, 448, 448])
torch.Size([101, 3, 448, 448])
torch.Size([101, 3, 448, 448])
torch.Size([63, 3, 448, 448])


In [7]:
for vid_clip in video_clip_set:
    for i,frame_id in enumerate(vid_clip['frame_id']):  # 遍历每一帧
        cur_img = cv2.imread("frames/"+str(vid_clip['frame_id'][i])+".jpg")
        for j in range(vid_clip['person_num']):  # 遍历每一个人
            gaze = vid_clip['gaze_p'+str(j)][i][0]
            head_bboxes = vid_clip['p'+str(j)][i]
            for xy in head_bboxes:
                xy = int(xy)
            head_center = [int(head_bboxes[1]+head_bboxes[3])//2,int(head_bboxes[0]+head_bboxes[2])//2]
            l = int(max(head_bboxes[3]-head_bboxes[1],head_bboxes[2]-head_bboxes[0])*1)
            gaze_len = l*1.0
            thick = max(5,int(l*0.01))
            print(gaze)
            cv2.arrowedLine(cur_img,(head_center[1],head_center[0]),
                        (int(head_center[1]-gaze_len*gaze[0]),int(head_center[0]-gaze_len*gaze[1])),
                        (230,253,11),thickness=thick)
        cv2.imwrite('new_frames/%d.jpg' % frame_id, cur_img)


[ 0.00945118  0.02937818 -0.9995237 ]
[ 0.2835158  -0.00776755 -0.9589361 ]
[ 3.1979585e-01 -4.4580299e-04 -9.4748634e-01]
[ 3.2799754e-01 -3.0867297e-05 -9.4467860e-01]
[ 0.33246917  0.00158205 -0.94311273]
[ 0.3379081   0.00545715 -0.9411633 ]
[ 0.36101297  0.0092432  -0.9325149 ]
[ 0.3522751   0.01017009 -0.93584126]
[ 0.35076836  0.01600446 -0.93632543]
[ 0.35114038  0.00446116 -0.93631226]
[ 0.34942198  0.0161883  -0.9368256 ]
[ 0.3475845   0.01632466 -0.9375066 ]
[ 0.3611136 -0.0285602 -0.9320844]
[ 0.34946987 -0.13666876 -0.9269264 ]
[ 0.34573802 -0.15753603 -0.92501223]
[ 0.3520656  -0.15874375 -0.9224154 ]
[ 0.35441417 -0.17321531 -0.91890544]
[ 0.35325775 -0.17864975 -0.91831   ]
[ 0.34841177 -0.18475972 -0.9189522 ]
[ 0.3430133  -0.17802723 -0.9223059 ]
[ 0.2769291  -0.18324178 -0.94325644]
[ 0.05088336 -0.13542953 -0.98947954]
[-0.05976248 -0.15242893 -0.9865059 ]
[-0.05116244 -0.15006676 -0.9873512 ]
[-0.11114653 -0.15482299 -0.98167014]
[-0.10401864 -0.15802614 -0.9819408

KeyboardInterrupt: 

In [12]:
img = cv2.imread('new_frames/0.jpg')  #读取第一张图片
fps = 25
imgInfo = img.shape
size = (imgInfo[1],imgInfo[0])  #获取图片宽高度信息
print(size)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
videoWrite = cv2.VideoWriter('new_video.mp4',fourcc,fps,size)# 根据图片的大小，创建写入对象 （文件名，支持的编码器，25帧，视频大小（图片大小））
 
files = os.listdir('new_frames/')
out_num = len(files)
for i in range(0,out_num):
    fileName = 'new_frames/'+str(i)+'.jpg'    #循环读取所有的图片,假设以数字顺序命名
    img = cv2.imread(fileName)
 
    videoWrite.write(img)# 将图片写入所创建的视频对象

videoWrite.release()

(1920, 800)
